# สกัดค่าความสูงและพิกัดสถานีเรดาร์เชียงราย
* โปรแกรมจะติดตั้งโปรแกรม และทำการสกัดค่าที่ฝังใน uf
* ไฟล์ uf.bz2 อาจพบปัญหาเปิดไม่ได้ในที่นี้ เนื่องจากต้อง rename จาก uf.bz2 ด้วยการตัด .bz2 ออกไป เนื่องจากไฟล์ไม่ได้ถูกบีบอัดมาจริงๆ


In [5]:
!pip install -q arm-pyart

In [24]:
# CRI Radar Data Processor with Auto-Rename for Misnamed Files
import os
import tempfile
import shutil
import pyart
from google.colab import drive, files

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Test PyART installation
try:
    print("✓ PyART imported successfully!")
    print(f"PyART version: {pyart.__version__}")
except ImportError as e:
    print(f"✗ Error importing PyART: {e}")
    print("Please restart runtime and try installing again")

def try_read_with_rename(file_path):
    """Try to read radar file, attempting rename if necessary"""

    # Strategy 1: Try reading directly first
    try:
        radar = pyart.io.read(file_path)
        print(f"✓ Direct read successful")
        return radar
    except Exception as e:
        print(f"⚠ Direct read failed: {str(e)[:60]}...")

    # Strategy 2: Try different PyART readers
    readers = [
        ('UF reader', pyart.io.read_uf),
        ('CFRadial reader', pyart.io.read_cfradial),
    ]

    for reader_name, reader_func in readers:
        try:
            radar = reader_func(file_path)
            print(f"✓ Success with {reader_name}")
            return radar
        except Exception as e:
            print(f"⚠ {reader_name} failed: {str(e)[:50]}...")

    # Strategy 3: Try renaming to different extensions
    print("Trying file rename strategies...")

    extensions_to_try = [
        '.uf',      # Standard UF
        '.UF',      # Uppercase UF
        '.raw',     # Raw radar data
        '.vol',     # Volume scan
        '',         # No extension
    ]

    with tempfile.TemporaryDirectory() as temp_dir:
        original_name = os.path.basename(file_path)
        base_name = os.path.splitext(os.path.splitext(original_name)[0])[0]  # Remove all extensions

        for ext in extensions_to_try:
            try:
                temp_file_path = os.path.join(temp_dir, f"{base_name}{ext}")
                shutil.copy2(file_path, temp_file_path)

                print(f"  Trying extension: {ext if ext else '(no extension)'}")
                radar = pyart.io.read(temp_file_path)
                print(f"✓ SUCCESS with extension: {ext if ext else '(no extension)'}")
                return radar

            except Exception as e:
                continue

    # Strategy 4: Check if it might be compressed but misnamed
    print("Checking for compression format mismatches...")

    # Try reading as if it's actually gzipped
    try:
        import gzip
        with tempfile.NamedTemporaryFile(suffix='.uf') as temp_file:
            with gzip.open(file_path, 'rb') as gz_file:
                temp_file.write(gz_file.read())
                temp_file.flush()

            radar = pyart.io.read(temp_file.name)
            print("✓ SUCCESS! File was actually gzip compressed")
            return radar
    except Exception:
        pass

    # If all strategies fail
    print("✗ All reading strategies failed")
    return None

def process_single_cri_file(file_path):
    """Process a single CRI radar file with robust error handling"""

    if not os.path.exists(file_path):
        print(f"✗ File not found: {file_path}")
        return None

    filename = os.path.basename(file_path)
    print(f"\nProcessing {filename}...")

    # Try to read the file with multiple strategies
    radar = try_read_with_rename(file_path)

    if radar is None:
        print(f"✗ Could not read {filename} with any method")
        return None

    try:
        # Extract radar information
        lon = round(float(radar.longitude['data'][0]), 6)
        lat = round(float(radar.latitude['data'][0]), 6)
        alt = round(float(radar.altitude['data'][0]), 1)
        el = round(float(radar.elevation['data'][0]), 2)

        # Get station name
        if filename.startswith('CRI'):
            station = filename.split('@')[0]
        else:
            station = filename.split('.')[0]

        radar_info = {"coords": (lon, lat, alt), "el": el}
        print(f"✓ {station}: {radar_info}")

        return {station: radar_info}

    except Exception as e:
        print(f"✗ Error extracting radar info from {filename}: {e}")
        return None

def process_cri_folder(folder_path):
    """Process all CRI radar files in a folder"""

    if not os.path.exists(folder_path):
        print(f"✗ Folder not found: {folder_path}")
        return {}

    radars = {}

    # Find all potential radar files
    radar_files = []
    for file in os.listdir(folder_path):
        # Look for common radar file patterns
        if (file.endswith('.uf') or file.endswith('.uf.bz2') or
            file.endswith('.UF') or file.endswith('.raw') or
            file.startswith('CRI') or 'radar' in file.lower()):
            radar_files.append(os.path.join(folder_path, file))

    print(f"Found {len(radar_files)} potential radar files in folder")

    # Process each file
    success_count = 0
    for file_path in radar_files:
        result = process_single_cri_file(file_path)
        if result:
            radars.update(result)
            success_count += 1

    print(f"\n✓ Successfully processed {success_count}/{len(radar_files)} files")
    return radars

def save_radar_dictionary(radar_dict, filename='radar_dictionary.py'):
    """Save radar dictionary to file and download"""
    if not radar_dict:
        print("No radar data to save!")
        return

    with open(filename, 'w', encoding='utf-8') as f:
        f.write("# CRI Radar Station Dictionary\n")
        f.write("# Generated from radar data files\n")
        f.write("# Original code by Assoc. Prof. Dr. Nattapon Mahavik, Naresuan University\n\n")
        f.write("radars = {\n")
        for name, data in radar_dict.items():
            f.write(f'    "{name}": {data},\n')
        f.write("}\n")

    print(f"\n✓ Radar dictionary saved to '{filename}'")
    print("Downloading file...")
    files.download(filename)

def check_file_details(file_path):
    """Check file details for debugging"""
    if os.path.exists(file_path):
        size = os.path.getsize(file_path)
        print(f"File: {os.path.basename(file_path)}")
        print(f"Size: {size:,} bytes ({size/1024/1024:.2f} MB)")

        # Check first few bytes
        try:
            with open(file_path, 'rb') as f:
                first_bytes = f.read(20)
                print(f"First 20 bytes (hex): {first_bytes.hex()}")
                if b'UF' in first_bytes[:10]:
                    print("✓ Contains 'UF' signature")
        except Exception as e:
            print(f"Error reading file: {e}")

# Main functions for different use cases

def process_specific_file():
    """Process the specific CRI file you mentioned"""
    file_path = "/content/drive/MyDrive/1shared_etc/0อบรมอาจารย์อังกูรเชียงราย/0radar_data/CRI240@202409091000.uf.bz2" #ปรับแก้ path ที่เก็บไฟล์ uf ในกุเกิ้ลไดร์ฟตรงนี้

    print("Processing specific CRI file...")
    print("="*50)

    # Show file details first
    check_file_details(file_path)

    result = process_single_cri_file(file_path)

    if result:
        print(f"\n=== SUCCESS! ===")
        print("radars = {")
        for name, data in result.items():
            print(f'    "{name}": {data},')
        print("}")

        save_radar_dictionary(result)
        return result
    else:
        print("Failed to process the file!")
        return {}

def process_radar_folder():
    """Process all radar files in the folder"""
    folder_path = "/content/drive/MyDrive/1shared_etc/0อบรมอาจารย์อังกูรเชียงราย/0radar_data/"

    print("Processing all radar files in folder...")
    print("="*50)

    radars = process_cri_folder(folder_path)

    if radars:
        print(f"\n=== SUCCESS! Processed {len(radars)} stations ===")
        print("radars = {")
        for name, data in radars.items():
            print(f'    "{name}": {data},')
        print("}")

        save_radar_dictionary(radars)
        return radars
    else:
        print("No radar data processed successfully!")
        return {}

def process_custom_path(file_or_folder_path):
    """Process a custom file or folder path"""
    if os.path.isfile(file_or_folder_path):
        print(f"Processing single file: {file_or_folder_path}")
        check_file_details(file_or_folder_path)
        result = process_single_cri_file(file_or_folder_path)
        if result:
            save_radar_dictionary(result)
        return result or {}
    elif os.path.isdir(file_or_folder_path):
        print(f"Processing folder: {file_or_folder_path}")
        radars = process_cri_folder(file_or_folder_path)
        if radars:
            save_radar_dictionary(radars)
        return radars
    else:
        print(f"✗ Path not found: {file_or_folder_path}")
        return {}

print("="*60)
print("ROBUST CRI RADAR PROCESSOR READY!")
print("="*60)
print("Features:")
print("- Auto-detects and handles misnamed files")
print("- Tries multiple reading strategies")
print("- Handles compressed and uncompressed files")
print("- Provides detailed error reporting")
print("="*60)
print("Available functions:")
print("1. process_specific_file()     - Process the specific CRI240 file")
print("2. process_radar_folder()      - Process all files in the radar folder")
print("3. process_custom_path('path') - Process custom file or folder path")
print("="*60)

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ PyART imported successfully!
PyART version: 2.1.0
ROBUST CRI RADAR PROCESSOR READY!
Features:
- Auto-detects and handles misnamed files
- Tries multiple reading strategies
- Handles compressed and uncompressed files
- Provides detailed error reporting
Available functions:
1. process_specific_file()     - Process the specific CRI240 file
2. process_radar_folder()      - Process all files in the radar folder
3. process_custom_path('path') - Process custom file or folder path


In [25]:
process_specific_file()

Processing specific CRI file...
File: CRI240@202409091000.uf.bz2
Size: 8,016,208 bytes (7.64 MB)
First 20 bytes (hex): 00001fd655460feb002e003c003c000100010001
✓ Contains 'UF' signature

Processing CRI240@202409091000.uf.bz2...
⚠ Direct read failed: Invalid data stream...
⚠ UF reader failed: Invalid data stream...
⚠ CFRadial reader failed: [Errno -51] NetCDF: Unknown file format: '/content...
Trying file rename strategies...
  Trying extension: .uf
✓ SUCCESS with extension: .uf
✓ CRI240: {'coords': (99.881593, 19.961471, 444.0), 'el': 1.09}

=== SUCCESS! ===
radars = {
    "CRI240": {'coords': (99.881593, 19.961471, 444.0), 'el': 1.09},
}

✓ Radar dictionary saved to 'radar_dictionary.py'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'CRI240': {'coords': (99.881593, 19.961471, 444.0), 'el': 1.09}}